#### Forward kinematics of `Panda` cabinet environment

In [ ]:
from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *

from package.panda_env import * 

from PyQt5.QtWidgets import QApplication
app = QApplication(sys.argv)

In [ ]:
xml_path = merge_mjcfs(
    included_mjcf_files=[
        '../../asset/floor/floor_white_gray.xml',
        './asset/panda_inspire/panda_inspire_site_added.xml',
        './asset/cabinet/cabinet_half_closed.xml',
        './asset/object/cylinder.xml',
    ],
    output_xml_path = 'xml/panda_cabinet_scene.xml',
)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=True)

In [ ]:
# Configuration
panda_joints          = get_panda_joint_names()
inspire_joints        = get_inspire_joint_names()
inspire_active_joints = env.get_active_among_joints(inspire_joints)
p_cylinder_offset0    = get_p_offset_palm_to_cylinder()
qactive_inspire0      = get_qactive_inspire()
q_inspire0            = get_q_inspire(env)
q_pandas              = get_q_pandas_cabinet()

# Initialize env
set_panda_cabinet_env(env,panda_joints,inspire_joints,q_pandas['init'],q_inspire0)

# Sliders
sliders = MultiSliderQtWidget(
    window_width  = 0.2,
    window_height = 0.4,
    label_texts   = env.active_joint_names,
    label_colors  = get_label_colors_from_joint_axis(env,env.active_joint_names),
    slider_vals   = env.get_qpos(joint_names=env.active_joint_names),
    slider_mins   = env.active_joint_mins,
    slider_maxs   = env.active_joint_maxs,
)

# Buttons
buttons = MultiRadioQtWidget(
    title         = "Set Panda joint position",
    label_texts   = ["Panda pose"],
    option_texts  = [["q_panda_init","q_panda_final"]],
    initial_texts = ["q_panda_init"],
    y_offset      = 0.45,
    window_width  = 0.2,
    window_height = 0.05,
)

# Loop
while env.is_viewer_alive():
    # Update
    qactive = sliders.get_values()
    qrev = env.get_qrev_from_qactive(qactive)
    env.forward(q=qrev,joint_names=env.rev_joint_names)

    # Buttons
    if buttons.is_toggled(label_text="Panda pose"):
        panda_pose = buttons.get_option("Panda pose")
        if panda_pose == "q_panda_init": 
            env.forward(q=q_pandas["init"],joint_names=panda_joints)
        if panda_pose == "q_panda_final": 
            env.forward(q=q_pandas["final"],joint_names=panda_joints)
        sliders.set_values(env.get_qpos(joint_names=env.active_joint_names))

    # Move attached cylinder
    T_palm = get_T_palm_panda_inspire(env)
    T_cylinder = view_in_world(T=p2t(p_cylinder_offset0),T_wl=T_palm)
    env.set_T('body_cylinder','base_body',T_cylinder)

    # Get contact info
    contact_info = env.get_contact_info()
    
    # Render
    env.plot_global_coordinate_axes()
    env.plot_contact_info()
    env.viewer_text_overlay('#contact','[%d]'%(contact_info['n_contact']))
    env.render()
    app.processEvents()

# Final image show
imshow(env.final_rgb_img)

# Close
sliders.close()
buttons.close()
print("Done.")